# Data Structures

In [2]:
from dataclasses import dataclass
from enum import StrEnum


class EditOperation(StrEnum):
    """Type of edit operation between source and target words."""

    KEEP = "K"
    REPLACE = "R"
    INSERT = "I"
    DELETE = "D"
    MERGE = "M"
    SPLIT = "S"


class AlignmentType(StrEnum):
    """Type of alignment."""

    WORD = "word"
    SENTENCE = "sentence"


@dataclass
class Alignment:
    """Represents an alignment between source and target word spans.

    Stores the alignment between a span of source words and a span of target
    words, along with the edit operation.
    """

    source_start: int
    source_end: int

    target_start: int
    target_end: int

    operation: EditOperation
    label: str | None = None
    alignment_type: AlignmentType = AlignmentType.WORD


@dataclass
class BackPointer:
    """Represents a back pointer in the DP table.

    Stores the operation and previous indices for backtracking through the
    dynamic programming table.
    """

    operation: EditOperation

    prev_i: int
    prev_j: int

# 1. Alignement

In [14]:
class Aligner:
    """Aligns two lists of words using dynamic programming."""

    INSERT_DELETE_COST = 1
    REPLACE_COST = 2
    MERGE_COST = 1
    SPLIT_COST = 1

    def align_words(self, source: str, target: str) -> list[Alignment]:
        source = str.split(source," ")
        target = str.split(target, " ")
        _, parent = self._build_dp(source, target)
        return self._backtrack(source, target, parent)

    def align_characters(self, source: str, target: str) -> list[Alignment]:
        """Aligns two strings at character level."""
        source_chars = list(source)
        target_chars = list(target)

        _, parent = self._build_dp(source_chars, target_chars)
        return self._backtrack(source_chars, target_chars, parent)

    def levenshtein(self, a: str, b: str) -> int:
        """Calculate the Levenshtein distance between two strings."""
        n = len(a)
        m = len(b)

        if n == 0:
            return m
        if m == 0:
            return n

        dp = [[0] * (m + 1) for _ in range(n + 1)]

        for i in range(n + 1):
            dp[i][0] = i

        for j in range(m + 1):
            dp[0][j] = j

        for i in range(1, n + 1):
            for j in range(1, m + 1):
                dp[i][j] = 1 + min(dp[i - 1][j], dp[i][j - 1], dp[i - 1][j - 1])

                if a[i - 1] == b[j - 1]:
                    dp[i][j] = dp[i - 1][j - 1]

        return dp[n][m]

    def _word_cost(self, source_word: str, target_word: str) -> float:
        """Computes the cost of replacing source_word with target_word."""
        if source_word == target_word:
            return 0.0

        distance = self.levenshtein(source_word, target_word)
        return distance

    def _merge_cost(self, left: str, right: str, target: str) -> float:
        """Computes the cost of merging left and right into target."""
        merged = left + right
        return self._word_cost(merged, target)

    def _merge_cost_multiple(self, source_words: list[str], target: str) -> float:
        """Computes the cost of merging multiple source words into one target."""
        merged = "".join(source_words)
        return self._word_cost(merged, target)

    def _split_cost(self, source: str, left: str, right: str) -> float:
        """Computes the cost of splitting source into left and right."""
        split_target = left + right
        return self._word_cost(source, split_target)

    def _split_cost_multiple(self, source: str, target_words: list[str]) -> float:
        """Computes the cost of splitting source into multiple target words."""
        split_target = "".join(target_words)
        return self._word_cost(source, split_target)

    def _build_dp(self, source: list[str], target: list[str]):
        """Builds the DP table and parent pointers for the given source and target."""
        n = len(source)
        m = len(target)

        dp = [[float("inf")] * (m + 1) for _ in range(n + 1)]
        parent = [[None] * (m + 1) for _ in range(n + 1)]
        dp[0][0] = 0

        for i in range(1, n + 1):
            dp[i][0] = dp[i - 1][0] + self.INSERT_DELETE_COST * len(source[i - 1])
            parent[i][0] = BackPointer(EditOperation.DELETE, i - 1, 0)

        for j in range(1, m + 1):
            dp[0][j] = dp[0][j - 1] + self.INSERT_DELETE_COST * len(target[j - 1])
            parent[0][j] = BackPointer(EditOperation.INSERT, 0, j - 1)

        for i in range(1, n + 1):
            for j in range(1, m + 1):
                # KEEP && REPLACE
                replace_cost = dp[i - 1][j - 1] + self._word_cost(
                    source[i - 1], target[j - 1]
                )
                if replace_cost < dp[i][j]:
                    dp[i][j] = replace_cost

                    op = (
                        EditOperation.KEEP
                        if source[i - 1] == target[j - 1]
                        else EditOperation.REPLACE
                    )
                    parent[i][j] = BackPointer(op, i - 1, j - 1)

                # DELETE
                delete_cost = dp[i - 1][j] + self.INSERT_DELETE_COST * len(
                    source[i - 1]
                )
                if delete_cost < dp[i][j]:
                    dp[i][j] = delete_cost

                    parent[i][j] = BackPointer(EditOperation.DELETE, i - 1, j)

                # INSERT
                insert_cost = dp[i][j - 1] + self.INSERT_DELETE_COST * len(
                    target[j - 1]
                )
                if insert_cost < dp[i][j]:
                    dp[i][j] = insert_cost

                    parent[i][j] = BackPointer(EditOperation.INSERT, i, j - 1)

                # # MERGE TODO: make it iterative for multiple merges
                # if i >= 2:
                #     merge_cost = dp[i - 2][j - 1] + self._merge_cost(
                #         source[i - 2], source[i - 1], target[j - 1]
                #     )

                #     if merge_cost < dp[i][j]:
                #         dp[i][j] = merge_cost

                #         parent[i][j] = BackPointer(EditOperation.MERGE, i - 2, j - 1)

                # # SPLIT TODO: make it iterative for multiple splits
                # if j >= 2:
                #     split_cost = dp[i - 1][j - 2] + self._split_cost(
                #         source[i - 1],
                #         target[j - 2],
                #         target[j - 1],
                #     )

                #     if split_cost < dp[i][j]:
                #         dp[i][j] = split_cost
                #         parent[i][j] = BackPointer(EditOperation.SPLIT, i - 1, j - 2)

                # MERGE
                best_merge_cost = float("inf")
                best_merge_k = 0
                for k in range(2, i + 1):
                    source_words = source[i - k : i]
                    merge_cost = dp[i - k][j - 1] + self._merge_cost_multiple(
                        source_words, target[j - 1]
                    )

                    if merge_cost < best_merge_cost:
                        best_merge_cost = merge_cost
                        best_merge_k = k
                    else:
                        break

                if best_merge_cost < dp[i][j]:
                    dp[i][j] = best_merge_cost
                    parent[i][j] = BackPointer(
                        EditOperation.MERGE, i - best_merge_k, j - 1
                    )

                # SPLIT
                best_split_cost = float("inf")
                best_split_k = 0
                for k in range(2, j + 1):
                    target_words = target[j - k : j]
                    split_cost = dp[i - 1][j - k] + self._split_cost_multiple(
                        source[i - 1], target_words
                    )

                    if split_cost < best_split_cost:
                        best_split_cost = split_cost
                        best_split_k = k
                    else:
                        break

                if best_split_cost < dp[i][j]:
                    dp[i][j] = best_split_cost
                    parent[i][j] = BackPointer(
                        EditOperation.SPLIT, i - 1, j - best_split_k
                    )

        return dp, parent

    def _backtrack(self, source, target, parent):
        """Backtrack through parent pointers to construct alignments.

        Backtracks through the parent pointers to construct the list of
        Alignment objects.
        """
        i = len(source)
        j = len(target)

        alignments = []
        while i > 0 or j > 0:
            ptr = parent[i][j]
            op = ptr.operation
            label = target[j - 1] if j > 0 else None

            if op in (EditOperation.KEEP, EditOperation.REPLACE):
                if op == EditOperation.KEEP:
                    label = None
                alignments.append(
                    Alignment(
                        source_start=i - 1,
                        source_end=i - 1,
                        target_start=j - 1,
                        target_end=j - 1,
                        operation=op,
                        label=label,
                    )
                )

            elif op == EditOperation.MERGE:
                alignments.append(
                    Alignment(
                        source_start=ptr.prev_i,
                        source_end=i - 1,
                        target_start=j - 1,
                        target_end=j - 1,
                        operation=op,
                    )
                )

            elif op == EditOperation.SPLIT:
                alignments.append(
                    Alignment(
                        source_start=i - 1,
                        source_end=i - 1,
                        target_start=ptr.prev_j,
                        target_end=j - 1,
                        operation=op,
                    )
                )

            elif op == EditOperation.DELETE:
                alignments.append(
                    Alignment(
                        source_start=i - 1,
                        source_end=i - 1,
                        target_start=j,
                        target_end=j - 1,
                        operation=op,
                    )
                )

            elif op == EditOperation.INSERT:
                alignments.append(
                    Alignment(
                        source_start=i,
                        source_end=i - 1,
                        target_start=j - 1,
                        target_end=j - 1,
                        operation=op,
                        label=label,
                    )
                )

            i = ptr.prev_i
            j = ptr.prev_j

        alignments.reverse()
        return alignments

trying some distance and similarity metrics to see how they perform on the task of aligning two sentences and extracting the edits between them.

In [4]:
from difflib import SequenceMatcher


def similarity(a: str, b: str) -> float:
    """Calculate the similarity ratio between two strings using SequenceMatcher."""
    return SequenceMatcher(
        None,
        a,
        b,
    ).ratio()


def common_prefix_ratio(a, b):
    """Calculate the ratio of common prefix length to max length."""
    count = 0

    for x, y in zip(a, b, strict=False):
        if x != y:
            break

        count += 1

    return count / max(len(a), len(b))

In [ ]:
source = "I am l o ve pythin"
target = "I ami love python i"

aligner = Aligner()

alignments = aligner.align_words(source, target)
print("word alignments:")
for alignment in alignments:
    print(alignment)

print("char alignments:")
alignments = aligner.align_characters(source, target)
for alignment in alignments:
    print(alignment)

word alignments:
Alignment(source_start=0, source_end=0, target_start=0, target_end=0, operation=<EditOperation.KEEP: 'K'>, label=None, alignment_type=<AlignmentType.WORD: 'word'>)
Alignment(source_start=1, source_end=1, target_start=1, target_end=1, operation=<EditOperation.REPLACE: 'R'>, label='ami', alignment_type=<AlignmentType.WORD: 'word'>)
Alignment(source_start=2, source_end=4, target_start=2, target_end=2, operation=<EditOperation.MERGE: 'M'>, label=None, alignment_type=<AlignmentType.WORD: 'word'>)
Alignment(source_start=5, source_end=5, target_start=3, target_end=3, operation=<EditOperation.REPLACE: 'R'>, label='python', alignment_type=<AlignmentType.WORD: 'word'>)
Alignment(source_start=6, source_end=5, target_start=4, target_end=4, operation=<EditOperation.INSERT: 'I'>, label='i', alignment_type=<AlignmentType.WORD: 'word'>)
char alignments:
Alignment(source_start=0, source_end=0, target_start=0, target_end=0, operation=<EditOperation.KEEP: 'K'>, label=None, alignment_type

# Edit Compression

In [6]:
class Extractor:
    """Extracts edit tags from word alignments."""

    def extract_tags(self, alignment: list[Alignment]) -> list[str]:
        """Compresses a word alignment into an edit tag string."""
        tags = list[str]()
        for a in alignment:
            if a.operation == EditOperation.KEEP:
                tags.append("k")
            elif a.operation == EditOperation.REPLACE:
                label = a.label
                if label is None:
                    raise ValueError("Label cannot be None for REPLACE operation")
                tags.append(f"r_[{label}]")

            elif a.operation == EditOperation.INSERT:
                label = a.label
                if label is None:
                    raise ValueError("Label cannot be None for INSERT operation")
                tags.append(f"i_[{label}]")

            elif a.operation == EditOperation.DELETE:
                tags.append("d")
            elif a.operation == EditOperation.MERGE:
                tags.append("m")
            elif a.operation == EditOperation.SPLIT:
                tags.append("s")
            else:
                raise ValueError(f"Unknown operation: {a.operation}")
        return tags

In [7]:
class Compressor:
    """Compresses tag strings by merging consecutive identical tags."""

    def compress_tags(self, tags: list[str]) -> str:
        """Compresses a tag string by merging consecutive identical tags."""
        if not tags:
            return ""

        compressed = []
        count = 1
        prev_tag = tags[0]

        for tag in tags[1:]:
            if tag == prev_tag:
                count += 1
            else:
                compressed.append(f"{prev_tag}*" if count > 1 else prev_tag)
                prev_tag = tag
                count = 1

        compressed.append(f"{prev_tag}*" if count > 1 else prev_tag)
        return "".join(compressed)

In [8]:
extractor = Extractor()
compressor = Compressor()
tags = extractor.extract_tags(alignments)
print(tags)
print(compressor.compress_tags(tags))
print(compressor.compress_tags(["k", "k", "d", "k", "r_[love]", "r_[love]", "k", "k"]))

['k', 'd', 'm', 's']
kdms
k*dkr_[love]*k*


# Subword Projection

In [9]:
class SubwordProjection:
    """Projects word-level alignments to subword level."""

    def tokenize(self, tokens: list[str]) -> list[str]:
        """Tokenizes a word into subwords."""
        # TODO: use preprocessing tokens when ready
        return ["سُم", "يه"]
        pass

    def compute_spans(self, subwords: list[str]) -> list[tuple[int, int]]:
        """Computes the character spans of each subword within the word."""
        # TODO: ARABert returns a ## for subwords, that needs to be handled (if i used
        # ARABert version of tokenization)
        # in span computation in case we used the ARABert version of
        # tokenization, otherwise, nope?

        current_ind = 0
        spans = []
        for subword in subwords:
            clean_subword = subword.replace("##", "")
            start = current_ind
            end = start + len(clean_subword) - 1
            spans.append((start, end))
            current_ind = end + 1
        return spans

    def find_corresponding_subword_edit(
        self, char_ind: int, spans: list[tuple[int, int]]
    ) -> int:
        """Finds the index of the subword that contains the given character index."""
        for i, (start, end) in enumerate(spans):
            if start <= char_ind <= end:
                return i
        raise ValueError(f"Character index {char_ind} out of bounds")

    def project(self, word: str, edits: list[Alignment]) -> list[list[Alignment]]:
        """Projects word-level edits to subword level."""
        subwords = self.tokenize(word)
        spans = self.compute_spans(subwords)
        projection = [[] for _ in subwords]

        for edit in edits:
            subword_ind = self.find_corresponding_subword_edit(edit.source_start, spans)
            projection[subword_ind].append(edit)
        return projection

    def compress_projection(
        self,
        projections: list[list[Alignment]],
        extractor: Extractor,
        compressor: Compressor,
    ) -> list[str]:
        """Compresses tags per subword for each word."""
        compressed_tags = []
        for projection in projections:
            tags = extractor.extract_tags(projection)
            compressed_tags.append(compressor.compress_tags(tags))
        return compressed_tags

In [10]:
sub_proj = SubwordProjection()
alignments = aligner.align_characters("سميه", "سُمية")
projection = sub_proj.project("سميه", alignments)
print(sub_proj.compress_projection(projection, extractor, compressor))

['ki_[ُ]k*', 'r_[ة]']
